# Fragrantica Perfume Scraper

This notebook contains a Python script using `requests` and `BeautifulSoup` to scrape specific perfume details from a Fragrantica product page. The script is designed to be easily reusable by changing the `target_url` variable.

**Note on Execution:** When run in certain environments (like the sandbox where this was developed), the script may encounter a **403 Forbidden** error due to the website's anti-bot measures. The script includes robust headers to mitigate this, but if the error persists, you may need to run it from a different network or consider using a proxy/VPN.

### Extracted Fields

| Field | Extraction Logic |
| :--- | :--- |
| `name` | Extracted from the main `<h1>` title, after removing the gender part. |
| `gender` | Extracted from the end of the main `<h1>` title (e.g., "for women and men"). |
| `rating` | Extracted from the "Perfume rating X.XX out of 5" text. |
| `rating_count` | Extracted from the "with X votes" text. |
| `main_accords` | Scraped from the "main accords" section, returning a list of strings. |
| `perfumers` | Scraped by looking for links or text containing the perfumer's name. |
| `description` | Extracted from the main descriptive paragraph on the page. |
| `url` | The input URL. |

In [ ]:
import json
import sys

sys.path.append("../..")  # repo root, so `common` (shared with streamlit_app/app.py) is importable

from common.scraping import scrape_fragrantica

In [ ]:
# DBTITLE 1,Add the scraped fragrance to frag_raw
# Builds one row matching the frag_raw table/CSV schema
# (name, gender, rating, rating_count, main_accords, perfumers, description, url)
# and MERGEs it in by url. From here, re-run clean_from_raw.py's cleaning
# query filtered to this url to populate fragrance_cleaned, then run
# generate_embeddings_for_new_frag.py to pick up the embedding.

from pyspark.sql import Row
from delta.tables import DeltaTable

if "error" in perfume_data:
    raise ValueError(f"Scrape failed, nothing to add: {perfume_data['error']}")

new_row = Row(
    name=str(perfume_data.get("name")),
    gender=str(perfume_data.get("gender")),
    rating=str(perfume_data.get("rating")),
    rating_count=str(perfume_data.get("rating_count")),
    main_accords=str(perfume_data.get("main_accords", [])),
    perfumers=str(perfume_data.get("perfumers", [])),
    description=str(perfume_data.get("description")),
    url=perfume_data["url"],
)
new_row_df = spark.createDataFrame([new_row])

target = DeltaTable.forName(spark, "fragrance_db.default.frag_raw")

(
    target.alias("t")
    .merge(new_row_df.alias("s"), "t.url = s.url")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Added/updated frag_raw row for: {perfume_data['url']}")

In [5]:
# Example Usage
target_url = "https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html"

print(f"Scraping data from: {target_url}\n")
perfume_data = scrape_fragrantica(target_url)

# Print the extracted data in a clean JSON format
print(json.dumps(perfume_data, indent=4, ensure_ascii=False))

# Optional: Print in the requested CSV-like format for easy comparison
print("\n--- CSV-like Output ---")
print("name\tgender\trating\trating_count\tmain_accords\tperfumers\tdescription\turl")

# Prepare the description for a single-line output (remove newlines/tabs)
clean_description = perfume_data.get('description', '').replace('\n', ' ').replace('\t', ' ').strip()

print(f"{perfume_data.get('name')}\t"
      f"{perfume_data.get('gender')}\t"
      f"{perfume_data.get('rating')}\t"
      f"{perfume_data.get('rating_count')}\t"
      f"{perfume_data.get('main_accords')}\t"
      f"{perfume_data.get('perfumers')}\t"
      f"{clean_description}\t"
      f"{perfume_data.get('url')}")

Scraping data from: https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html

{
    "url": "https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html",
    "name": "Velvet Rouge Fragrance World",
    "gender": "for women and men",
    "rating": "N/A",
    "rating_count": "N/A",
    "main_accords": [
        "rose"
    ],
    "perfumers": [],
    "description": "N/A"
}

--- CSV-like Output ---
name	gender	rating	rating_count	main_accords	perfumers	description	url
Velvet Rouge Fragrance World	for women and men	N/A	N/A	['rose']	[]	N/A	https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html
